<div style="background: #86d1f1ff; border-radius: 5px; padding: 1rem; margin-bottom: 1rem">
<img src="https://store.utec.edu.pe/files/Recursos/logo-utec-h.png" alt="Banner" width="150" />   
<div style="font-weight: bold; color: #434549ff; float: right "><u style="font-size: 28px;">Base de Datos II</u> <br />
<span style="float:right"> Profesor Heider Sanchez</span> <br /> 
<span style="float:right">  2026 - 1 </span>   
</div> </div>

# Laboratorio 8.2: Similitud de Coseno e Indice Invertido

> **Prof. Heider Sanchez**  

## Introducción

Este laboratorio extiende las capacidades desarrolladas en el laboratorio 7.1, enfocándose en técnicas avanzadas de recuperación de información. Utilizaremos los Bag of Words previamente generados para implementar dos funcionalidades esenciales en los motores de búsqueda modernos: el **Índice Invertido** para recuperación eficiente de documentos, y la **Similitud de Coseno** para resultados ordenados por relevancia.


### Objetivos
- Implementar la conexión a PostgreSQL para extraer id, contenido y bag_of_words de los documentos.
- Construir el índice invertido (posting lists), calcular estadísticas IDF y la norma vectorial de cada documento.
- Implementar búsquedas booleanas (AND, OR y AND-NOT) con complejidad O(n+m) usando el índice invertido.
- Implementar búsquedas rankeadas usando la Similitud de Coseno:
  - Procesar consultas en lenguaje natural aplicando tokenización y cálculo del TF.
  - Utilizar el índice invertido para obtener los documentos que intersectan con la query.
  - Calcular similitud de coseno con TF-IDF entre la query y los documentos recuperados.
  - Devolver los top-k resultados ordenados por relevancia.
  - **Evitar usar la representación vectorial dispersa.**

In [2]:
import psycopg2
import pandas as pd
import json

def connect_db():
    conn = psycopg2.connect(
        dbname="Information Retrieval",
        user="postgres",
        password="123456",
        host="localhost",
        port="5432"
    )
    return conn

def fetch_data():
    conn = connect_db()
    query = "SELECT id, contenido, bag_of_words FROM noticias;"
    df = pd.read_sql(query, conn)
    # acá solo comente esa línea porque ya tenemos el bag_of_words como un diccionario en la base de datos pq en el anteiror lab lo guardaron así en la parte 2 y 3 xd
    #df['bag_of_words'] = df['bag_of_words'].apply(json.loads) 
    conn.close()
    return df

noticias_df = fetch_data()

C:\Users\josue\AppData\Local\Temp\ipykernel_51028\3272244204.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [4]:
noticias_df

,id,contenido,bag_of_words
0,372,Durante el foro La banca articulador empresari...,"{'si': 1, 'aca': 1, 'adn': 1, 'cas': 1, 'dej':..."
1,373,El regulador de valores de China dijo el domin...,"{'dat': 1, 'deb': 1, 'hab': 1, 'inc': 1, 'mas'..."
2,374,En una industria históricamente masculina como...,"{'go': 1, 'mi': 1, 'air': 1, 'asi': 1, 'baj': ..."
3,375,Con el dato de marzo el IPC interanual encaden...,"{'agu': 1, 'asi': 1, 'car': 2, 'dat': 2, 'ine'..."
4,376,Ayer en Cartagena se dio inicio a la versión n...,"{'agu': 1, 'asi': 1, 'aun': 1, 'cup': 2, 'deb'..."
...,...,...,...
1212,1582,La XVI Cumbre de la Alianza del Pacífico se ll...,"{'bah': 1, 'bas': 1, 'cab': 1, 'dam': 1, 'gan'..."
1213,1583,La inflación en España volvió a dispararse en ...,"{'beb': 1, 'dat': 1, 'ine': 1, 'ipc': 2, 'lun'..."
1214,1585,La espiral alcista de los precios continúa y g...,"{'com': 2, 'dud': 1, 'dur': 1, 'hab': 1, 'jef'..."
1215,1586,Las grandes derrotas nacionales son experienci...,"{'xi': 1, 'asi': 1, 'cas': 1, 'dej': 1, 'gas':..."


In [21]:
def fetch_stopwords():
    conn = connect_db()
    query = "SELECT word FROM stopwords;"
    df_words = pd.read_sql(query, conn)
    conn.close()
    return set(df_words['word'].tolist())

stopwords_es = fetch_stopwords()
print(f"¡Listo! Se cargaron {len(stopwords_es)} stopwords desde la base de datos.")

¡Listo! Se cargaron 608 stopwords desde la base de datos.


C:\Users\josue\AppData\Local\Temp\ipykernel_51028\3088269216.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_words = pd.read_sql(query, conn)


In [22]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

# Inicializamos el stemmer en español para reducir las palabras a su raíz léxica
stemmer = SnowballStemmer('spanish')

def preprocess(text):
    """
    Recibe un texto crudo y realiza: minúsculas, tokenización, 
    eliminación de stopwords y stemming.
    """
    if not text:
        return []
    
    # Convertir el texto a minúscula
    text = text.lower() 
    
    # Tokenizar las palabras
    tokens = word_tokenize(text) 
    
    # Eliminación de stopwords, números y símbolos
    cleaned_tokens = [
        token for token in tokens 
        if token.isalpha() and token not in stopwords_es
    ] 
    
    # Stemming (reducir las palabras a su raíz)
    stemmed_tokens = [stemmer.stem(token) for token in cleaned_tokens] 
    
    return stemmed_tokens




## 1. (6 puntos) Construcción del Indice Invertido 

A partir de los  `bag of words` almacenados en la base de datos  (Laboratorio 8.1), se debe construir un índice invertido y conservarlo en un diccionario de Python para su eficiente recuperación.

In [5]:
noticias_df = fetch_data()

C:\Users\josue\AppData\Local\Temp\ipykernel_51028\3272244204.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [ ]:
import math
import re
from collections import Counter

class InvertedIndex:
    def __init__(self):
        self.index = {}
        self.idf = {}
        self.length = {}
        self.df = None
        self.N = 0

    def build_from_db(self):
        # Leer desde PostgreSQL todos los bag of words
        # Construir el índice invertido, el idf y la norma (longitud) de cada documento
        
        """
        indice  = {
            "word1": [("doc1", tf1), ("doc2", tf2), ("doc3", tf3)],
            "word2": [("doc2", tf2), ("doc4", tf4)],
            "word3": [("doc3", tf3), ("doc5", tf5)],
        } 
        idf  = {
            "word1": 3,
            "word2": 2,
            "word3": 2,
        } 
        length = {
            "doc1": 15.5236,
            "doc2": 10.5236,
            "doc3": 5.5236,
        }
        """
        df = fetch_data()
        self.N = len(df)
        df_counts = {}
        self.df = df
        
        # Contamos en cuántos documentos aparece cada palabra (df)
        for _, row in df.iterrows():
            bow = row['bag_of_words']
            for word in bow.keys():
                df_counts[word] = df_counts.get(word, 0) + 1
                
        # Calculamos idf para cada término (log10(N / df))
        for word, count in df_counts.items():
            self.idf[word] = math.log10(self.N / count)
            
        # Construimos el índice invertido y las normas de los documentos
        for _, row in df.iterrows():
            doc_id = row['id']
            bow = row['bag_of_words']
            
            doc_length_sq = 0.0
            
            for word, tf in bow.items():
                if word not in self.index:
                    self.index[word] = []
                self.index[word].append((doc_id, tf))
                # Calcular peso TF-IDF para la norma del documento
                tf_weight = 1 + math.log10(tf) if tf > 0 else 0
                w_td = tf_weight * self.idf[word]
                
                doc_length_sq += w_td ** 2
                
            self.length[doc_id] = math.sqrt(doc_length_sq)
        pass
    
    def L(self, word):
        processed = preprocess(word)

        if len(processed) == 0:
            return []

        term = processed[0]
        return self.index.get(term, [])

    def cosine_search(self, query, top_k=5):  
        score = {}
        # No es necesario usar vectores numericos del tamaño del vocabulario
        # Guiarse del algoritmo visto en clase
        # Se debe calcular el tf-idf de la query y de cada documento
        # La query se debe procesar  al igual como se procesaron los documentos (bag of words)
        
        query = re.sub(r'[^\w\s]', '', query.lower())
        query_bow = Counter(query.split())
        query_length_sq = 0.0
        for word, q_tf in query_bow.items():
            if word in self.idf:
                q_tf_weight = 1 + math.log10(q_tf) if q_tf > 0 else 0
                w_tq = q_tf_weight * self.idf[word]
                
                query_length_sq += w_tq ** 2
                
                posting_list = self.L(word) # lista de documentos que contienen a word 
                
                for doc_id, doc_tf in posting_list:
                    doc_tf_weight = 1 + math.log10(doc_tf) if doc_tf > 0 else 0
                    w_td = doc_tf_weight * self.idf[word]
                    
                    # Acumulamos el producto punto
                    score[doc_id] = score.get(doc_id, 0.0) + (w_tq * w_td)
                    
        query_length = math.sqrt(query_length_sq)
        
        # Normalización Coseno: dividir por las normas
        if query_length > 0:
            for doc_id in score:
                score[doc_id] = score[doc_id] / (query_length * self.length[doc_id])
        
        # Ordenar el score resultante de forma descendente
        result = sorted(score.items(), key= lambda tup: tup[1], reverse=True)
        # retornamos los k documentos mas relevantes (de mayor similitud a la query)
        return result[:top_k] 
    
    #this implementation is going to be used for the question 3:
    def showDocument(self, doc_id):
        # Retorna el contenido del documento dado su id
        conn = connect_db()
        query = "SELECT contenido FROM noticias WHERE id = %s;"
        cursor = conn.cursor()
        cursor.execute(query, (doc_id,))
        result = cursor.fetchone()
        conn.close()
        return result[0] if result else None 

    def showDocuments(self, doc_ids):
        result = []

        for item in doc_ids:
            if isinstance(item, tuple):
                doc_id = item[0]
            else:
                doc_id = item

            result.append((doc_id, self.showDocument(doc_id)))

        return result
    

In [24]:
#Esto es mi prueba si quieren lo pueden borrar despues
buscador = InvertedIndex()
buscador.build_from_db()
resultados = buscador.cosine_search("dol ia cas", top_k=5)
print(resultados)

[(1295, 0.19005712212091216), (544, 0.12379060061643506), (1389, 0.12133700566465115), (685, 0.1070670234830583), (857, 0.1070670234830583)]


C:\Users\josue\AppData\Local\Temp\ipykernel_51028\3272244204.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


## 2. (6 puntos) Consultas Booleanas usando el indice invertido

Implementar búsquedas booleanas utilizando el índice invertido construido anteriormente. La búsqueda debe:

- Soportar los operadores básicos:
    - AND: intersección de documentos
    - OR: unión de documentos
    - AND-NOT: diferencia de documentos
- Procesar consultas como:
    - "sostenibilidad AND ambiente AND renovable"
    - "tecnología AND (banca OR finanzas)"
    - "economía AND-NOT inflación"    

####  Pruebas funcionales

In [28]:
idx = InvertedIndex()
idx.build_from_db()

def get_doc_id(item):
    if isinstance(item, tuple):
        return item[0]
    return item


def AND(list1, list2):
    i, j = 0, 0
    result = []

    list1 = sorted(list1, key=get_doc_id)
    list2 = sorted(list2, key=get_doc_id)

    while i < len(list1) and j < len(list2):
        doc1 = get_doc_id(list1[i])
        doc2 = get_doc_id(list2[j])

        if doc1 == doc2:
            result.append(doc1)
            i += 1
            j += 1
        elif doc1 < doc2:
            i += 1
        else:
            j += 1

    return result


def OR(list1, list2):
    i, j = 0, 0
    result = []

    list1 = sorted(list1, key=get_doc_id)
    list2 = sorted(list2, key=get_doc_id)

    while i < len(list1) and j < len(list2):
        doc1 = get_doc_id(list1[i])
        doc2 = get_doc_id(list2[j])

        if doc1 == doc2:
            result.append(doc1)
            i += 1
            j += 1
        elif doc1 < doc2:
            result.append(doc1)
            i += 1
        else:
            result.append(doc2)
            j += 1

    while i < len(list1):
        result.append(get_doc_id(list1[i]))
        i += 1

    while j < len(list2):
        result.append(get_doc_id(list2[j]))
        j += 1

    return result


def AND_NOT(list1, list2):
    i, j = 0, 0
    result = []

    list1 = sorted(list1, key=get_doc_id)
    list2 = sorted(list2, key=get_doc_id)

    while i < len(list1) and j < len(list2):
        doc1 = get_doc_id(list1[i])
        doc2 = get_doc_id(list2[j])

        if doc1 == doc2:
            i += 1
            j += 1
        elif doc1 < doc2:
            result.append(doc1)
            i += 1
        else:
            j += 1

    while i < len(list1):
        result.append(get_doc_id(list1[i]))
        i += 1

    return result

C:\Users\josue\AppData\Local\Temp\ipykernel_51028\3272244204.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [30]:
# Prueba 1
result = AND(idx.L("sostenibilidad"), AND(idx.L("ambiente"), idx.L("renovables")))
print("sostenibilidad AND ambiente AND renovable: ", idx.showDocuments(result))

    

sostenibilidad AND ambiente AND renovable:  [(423, 'La importancia que cada día más empresas le otorgan al tema de la sostenibilidad como parte esencial de su modelo de negocio está cambiando las estructuras corporativas. Esto se debe a la urgencia que existe en la materia en todo el mundo. Para conocer con mayor detalle los esfuerzos que se realizan en México, se llevó a cabo un panel sobre sostenibilidad como parte de Diálogos BBVA 2022: Talento hecho en México.\r\nEl panel “Relevancia de las empresas e instituciones en la transformación de sociedades sostenibles”, moderado por Sergio Torres Lebrija, director de Estrategia e Innovación de la Banca Digital y Sostenibilidad de BBVA México, contó con la participación de María José Treviño, directora general de Acclaim Energy México; Luis Darío Ochoa Rodríguez, director de Sostenibilidad y Licencia Social de Coca-Cola FEMSA (KOF); y de Carlos Mendieta Zerón, presidente del Pacto Mundial México y director de sustentabilidad en PetStar.\r\

In [31]:
# Prueba 2
result = AND(idx.L("tecnología"), OR(idx.L("banca"), idx.L("finanzas")))
print("tecnología AND (banca OR finanzas): ", idx.showDocuments(result))


tecnología AND (banca OR finanzas):  [(383, "BBVA ha presentado BBVA Spark, su propuesta integral de servicios financieros para acompañar a las compañías innovadoras en sus distintas fases de crecimiento. Spark permitirá a estas compañías cubrir todas sus necesidades financieras en un mismo lugar, así como contar con productos de financiación sofisticados como el ‘venture debt’ o los ‘growth loans’. Para ello, BBVA Spark pondrá a su disposición un modelo de relación diferencial con un equipo de especialistas que hablan su propio idioma.\r\nInnovación\r\nBBVA ha adelantado durante la jornada de clausura de South Summit que está preparando una completa propuesta de servicios financieros adaptados a los distintos momentos de crecimiento de las ‘startups’ y ‘scaleups’. “Queremos ser el banco de las empresas de alto crecimiento y financiarles”, aseguró Roberto Albaladejo, responsable de Banking for Growth Companies, quien añadió que existe una “gran oportunidad” en ayudarles y construir con

In [32]:
# Prueba 3
result = AND_NOT(idx.L("economía"), idx.L("inflación"))
print("economía AND-NOT inflación: " , idx.showDocuments(result))

economía AND-NOT inflación:  [(385, 'Las plataformas de domicilios fueron uno de los jugadores más importantes de la pandemia pero acabando el confinamiento es hora de pasar la página y enfocarse en los temas de fondo del sector. Felipe Ossa CEO de Domicilios.com dijo que es muy bueno que se empiece a hablar de regulación aunque dijo que si Colombia no resuelve rápido las reglas de juego el país se quedará rezagado.Cuánto crecieron en pandemia es sostenibleTodas las plataformas crecieron sobre todo al inicio de la cuarentena. En el momento en que las personas empiezan a salir de sus casas se ve un repunte de los restaurantes lo que ha tenido un impacto negativo para las plataformas y pero se agrandó la base de usuarios y hay una base sobre la cual construir.Ahora es muy difícil prever qué va a ocurrir porque vamos de la mano de las definiciones del Gobierno y vemos que países que van más adelantados en la pandemia han vuelto a cerrar.Qué planes tienenHablar de planes en este momento es

In [33]:
# Prueba 4
# Consulta: banco AND tecnología AND-NOT inflación
result = AND_NOT(
    AND(idx.L("banco"), idx.L("tecnología")),
    idx.L("inflación")
)

print("banco AND tecnología AND-NOT inflación: ", idx.showDocuments(result))

banco AND tecnología AND-NOT inflación:  [(383, "BBVA ha presentado BBVA Spark, su propuesta integral de servicios financieros para acompañar a las compañías innovadoras en sus distintas fases de crecimiento. Spark permitirá a estas compañías cubrir todas sus necesidades financieras en un mismo lugar, así como contar con productos de financiación sofisticados como el ‘venture debt’ o los ‘growth loans’. Para ello, BBVA Spark pondrá a su disposición un modelo de relación diferencial con un equipo de especialistas que hablan su propio idioma.\r\nInnovación\r\nBBVA ha adelantado durante la jornada de clausura de South Summit que está preparando una completa propuesta de servicios financieros adaptados a los distintos momentos de crecimiento de las ‘startups’ y ‘scaleups’. “Queremos ser el banco de las empresas de alto crecimiento y financiarles”, aseguró Roberto Albaladejo, responsable de Banking for Growth Companies, quien añadió que existe una “gran oportunidad” en ayudarles y construir

In [34]:
# Prueba 5
# Consulta: sostenibilidad AND (finanzas OR banca) AND-NOT inflación
result = AND_NOT(
    AND(
        idx.L("sostenibilidad"),
        OR(idx.L("finanzas"), idx.L("banca"))
    ),
    idx.L("inflación")
)

print("sostenibilidad AND (finanzas OR banca) AND-NOT inflación: ", idx.showDocuments(result))

sostenibilidad AND (finanzas OR banca) AND-NOT inflación:  [(372, 'Durante el foro La banca articulador empresarial para el desarrollo sostenible el director de sostenibilidad y clientes globales de BBVA en Colombia Andrés García aseguró que es importante entender que la sostenibilidad no la podemos asociar a mayores costos. Yo creo que el no tener un concepto de negocio sostenible puede tener un mayor impacto de lo que imaginamos.Para García el reto más importante es no cambiar prioridades ni que compitan entre sí necesariamente. En muchos de los casos se debe tratar de mantener la prioridad en cuanto a la ambición de negocios más sostenibles un reto enorme por la coyuntura. La sostenibilidad nos abre oportunidades a mejores fuentes de financiamiento agregó.El directivo argumentó que lo que se encuentra en juego acá no es un tema de rentabilidad o de negocios en particular es un tema de viabilidad del mundo de los negocios y del mundo físico en general como lo conocemos. Además los ri

## 3. (8 puntos) Similitud de Coseno usando el indice invertido
Implementar búsqueda por similitud de coseno aprovechando el índice invertido:

- Proceso de búsqueda:
    - Recibe una consulta en lenguaje natural y un parámetro top_k
    - Utiliza el índice invertido para identificar documentos candidatos
    - Calcula similitud de coseno solo con los documentos relevantes utilizando los pesos TF-IDF
    - Retorna los top-k documentos más similares

<img src="https://1drv.ms/i/c/0c2923df9f1f816f/IQSELMi5qcbqS7lsy5sn8ZLpAZ3G2ciXdabecVJ0vhKoL78" width="500" align="" />

####  Pruebas funcionales

In [ ]:
test_queries = [
    "¿Cuáles son las últimas innovaciones en la banca digital y la tecnología financiera?",
    "evolución de la inflación y el crecimiento de la economía en los últimos años",
    "avances sobre sostenibilidad y energías renovables para el medio ambiente"
]

for test in test_queries:    
    results = idx.cosine_search(test['query'], test['top_k'])
    print(f"Top {test['top_k']} documentos más similares:")    
    for doc_id, score in results:
        print(f"Doc {doc_id}: {score:.3f}: ", idx.showDocument(doc_id))